In [1]:
import datasets
import huggingface_hub

print("datasets:", datasets.__version__)
print("huggingface_hub:", huggingface_hub.__version__)

datasets: 4.0.0
huggingface_hub: 1.20.1


In [2]:
!pip uninstall -y datasets huggingface_hub
!pip install datasets==3.6.0 huggingface_hub==0.34.4

Found existing installation: datasets 4.0.0
Uninstalling datasets-4.0.0:
  Successfully uninstalled datasets-4.0.0
Found existing installation: huggingface_hub 1.20.1
Uninstalling huggingface_hub-1.20.1:
  Successfully uninstalled huggingface_hub-1.20.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 24.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.12.1 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 0.34.4 which is incompatible.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.34.4 which is incompatible.


In [1]:
from datasets import load_dataset

dataset = load_dataset("ag_news")
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})


In [2]:
X_train = dataset["train"]["text"]
y_train = dataset["train"]["label"]

X_test = dataset["test"]["text"]
y_test = dataset["test"]["label"]

print(X_train[0])
print("Label:", y_train[0])

Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.
Label: 2


In [4]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_words = 20000
max_length = 100

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train = tokenizer.texts_to_sequences(X_train)
X_test = tokenizer.texts_to_sequences(X_test)

X_train = pad_sequences(X_train, maxlen=max_length, padding="post")
X_test = pad_sequences(X_test, maxlen=max_length, padding="post")

In [5]:
from tensorflow.keras.utils import to_categorical

y_train = to_categorical(y_train, num_classes=4)
y_test = to_categorical(y_test, num_classes=4)

In [6]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout

model = Sequential()

model.add(Embedding(input_dim=max_words,
                    output_dim=128,
                    input_length=max_length))

model.add(SimpleRNN(64))

model.add(Dropout(0.5))

model.add(Dense(32, activation="relu"))

model.add(Dense(4, activation="softmax"))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [7]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [8]:
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.2
)

Epoch 1/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 79s 102ms/step - accuracy: 0.5162 - loss: 1.1134 - val_accuracy: 0.7092 - val_loss: 0.8705
Epoch 2/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 70s 87ms/step - accuracy: 0.6469 - loss: 0.9465 - val_accuracy: 0.6864 - val_loss: 0.9182
Epoch 3/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 66s 88ms/step - accuracy: 0.7222 - loss: 0.8448 - val_accuracy: 0.6918 - val_loss: 0.9012
Epoch 4/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 80s 85ms/step - accuracy: 0.7774 - loss: 0.7263 - val_accuracy: 0.7248 - val_loss: 0.8396
Epoch 5/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 83s 86ms/step - accuracy: 0.8035 - loss: 0.6536 - val_accuracy: 0.7221 - val_loss: 0.8391


In [9]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

238/238 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.7547 - loss: 0.7615
Test Loss: 0.7614963054656982
Test Accuracy: 0.7547368407249451


In [10]:
import numpy as np
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

y_pred = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test, axis=1)

print("Accuracy:", accuracy_score(y_true, y_pred))

print(classification_report(
    y_true,
    y_pred,
    target_names=["World", "Sports", "Business", "Sci/Tech"]
))

238/238 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step
Accuracy: 0.7547368421052632
              precision    recall  f1-score   support

       World       0.80      0.76      0.78      1900
      Sports       0.80      0.83      0.82      1900
    Business       0.79      0.63      0.70      1900
    Sci/Tech       0.66      0.79      0.72      1900

    accuracy                           0.75      7600
   macro avg       0.76      0.75      0.75      7600
weighted avg       0.76      0.75      0.75      7600



In [11]:
news = [
    "India won the cricket world cup after defeating Australia in the final."
]

sequence = tokenizer.texts_to_sequences(news)

sequence = pad_sequences(sequence,
                         maxlen=max_length,
                         padding="post")

prediction = model.predict(sequence)

classes = [
    "World",
    "Sports",
    "Business",
    "Sci/Tech"
]

print("Predicted Category:", classes[np.argmax(prediction)])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
Predicted Category: Sports
